# Chemical Structure Data Cleaning Pipeline

This notebook standardizes and cleans a literature-derived compound dataset. The pipeline:
1. Retrieves canonical SMILES from PubChem by compound name
2. Filters out non-carbon-containing structures
3. Handles salt forms by stripping counterions and selecting the largest fragment
4. Removes stereochemistry, neutralizes charges, and canonicalizes tautomers
5. Applies PubChem-based structure standardization in batches

## 1. Imports

In [ ]:
import time

import numpy as np
import pandas as pd
import pubchempy as pcp
import requests
import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, PandasTools, SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

from standardizeUtils.standardizeUtils import standardize_structure_list_with_pubchem

# Suppress RDKit warnings for cleaner output
RDLogger.DisableLog('rdApp.*')

## 2. Helper Functions

In [ ]:
def get_SMILES_by_name(name):
    """Query PubChem for the canonical SMILES of a compound by its name.
    Returns np.nan if the compound cannot be found.
    """
    compounds = pcp.get_compounds(name, 'name')
    try:
        smiles = compounds[0].connectivity_smiles
    except IndexError:
        print(f"Warning: SMILES for '{name}' could not be retrieved from PubChem. Inserting NaN.")
        smiles = np.nan
    return smiles


def contains_carbon(romol):
    """Return True if the RDKit molecule contains at least one carbon atom."""
    atomic_nums = [atom.GetAtomicNum() for atom in romol.GetAtoms()]
    return 6 in atomic_nums


def is_disconnected(romol):
    """Return True if the molecule consists of more than one disconnected fragment
    (e.g., a salt or multi-component mixture).
    """
    fragments = rdkit.Chem.rdmolops.GetMolFrags(romol, asMols=True)
    return len(fragments) > 1


def remove_fragment(romol, smarts):
    """Strip a specific fragment (defined by a SMARTS string) from a molecule
    using RDKit's SaltRemover.
    """
    remover = SaltRemover.SaltRemover(defnData=smarts)
    return remover.StripMol(romol)


def keep_largest_fragment(romol):
    """For a disconnected molecule, retain only the largest fragment by atom count.
    This is used as a final salt-removal step after targeted fragment stripping.
    """
    fragments = rdkit.Chem.rdmolops.GetMolFrags(romol, asMols=True)
    return max(fragments, key=lambda m: m.GetNumAtoms())


def neutralize_atoms(romol):
    """Neutralize formally charged atoms where possible by adjusting hydrogen counts.
    Targets atoms that carry a charge but are not part of a zwitterion-stabilizing pair.
    Returns the original molecule unchanged if neutralization fails.
    """
    try:
        pattern = Chem.MolFromSmarts(
            "[+1!h0!$([*]~[-1,-2,-3,-4]),-1!$([*]~[+1,+2,+3,+4])]"
        )
        matches = [match[0] for match in romol.GetSubstructMatches(pattern)]
        for idx in matches:
            atom = romol.GetAtomWithIdx(idx)
            charge = atom.GetFormalCharge()
            h_count = atom.GetTotalNumHs()
            atom.SetFormalCharge(0)
            atom.SetNumExplicitHs(h_count - charge)
            atom.UpdatePropertyCache()
    except Exception:
        pass
    return romol


def standardize_in_batches(smiles_list, initial_batch_size=1000):
    """Standardize a list of SMILES strings via PubChem in batches.

    If a batch request fails (e.g., due to a network error), the batch size is
    halved and the request is retried. Batches that cannot be processed even at
    the minimum size (100) are skipped with a warning.

    Parameters
    ----------
    smiles_list : list of str
        Input SMILES strings to standardize.
    initial_batch_size : int, optional
        Number of SMILES to submit per request (default: 1000).

    Returns
    -------
    list of str
        Standardized SMILES strings in the same order as the input.
    """
    total = len(smiles_list)
    standardized = []
    idx = 0

    while idx < total:
        batch_size = initial_batch_size
        success = False

        while batch_size >= 100:
            try:
                print(f"Processing records {idx}–{min(idx + batch_size, total)} "
                      f"(batch size={batch_size})")
                batch = smiles_list[idx: idx + batch_size]
                standardized_batch = standardize_structure_list_with_pubchem(batch, 'smiles')
                standardized.extend(standardized_batch)
                idx += batch_size
                success = True
                break

            except requests.exceptions.RequestException as e:
                print(f"Request failed: {e}. Reducing batch size and retrying...")
                batch_size //= 2
                time.sleep(2)

        if not success:
            print(f"Skipping records {idx}–{idx + batch_size}: repeated request failures.")
            idx += batch_size

    return standardized

## 3. Load Raw Data

In [ ]:
data = pd.read_csv("your_data_with_compound_names.csv")
print(f"Loaded {len(data)} rows.")
data.head()

## 4. Retrieve SMILES from PubChem

In [ ]:
# Fetch canonical SMILES for each compound by name via PubChem
data['SMILES'] = data['Name'].apply(get_SMILES_by_name)
print(f"SMILES retrieved: {data['SMILES'].notna().sum()} / {len(data)}")

In [ ]:
#remove rows where SMILES could not be retrieved
data = data[data['SMILES'].isnull() == False].reset_index(drop=True)

## 5. Generate RDKit Molecule Objects

In [ ]:
# Convert SMILES to RDKit ROMol objects; rename raw SMILES column for tracking
PandasTools.AddMoleculeColumnToFrame(data, 'SMILES', 'ROMol')
data.rename(columns={'SMILES': 'SMILES_raw'}, inplace=True)

## 6. Filter: Remove Non-Carbon-Containing Structures

In [ ]:
# Flag structures that contain at least one carbon atom
data['contains_carbon'] = data['ROMol'].apply(contains_carbon)
n_removed = (~data['contains_carbon']).sum()
print(f"Removing {n_removed} inorganic/non-carbon structures.")

In [ ]:
# Retain only carbon-containing compounds and regenerate ROMol column
data = data[data['contains_carbon']].reset_index(drop=True)
PandasTools.AddMoleculeColumnToFrame(data, 'SMILES_raw', 'ROMol')
print(f"{len(data)} rows remaining after carbon filter.")

## 7. Salt Handling

Disconnected structures (salts, hydrates, co-crystals) are identified, stripped of
common counterions and solvents, and reduced to their largest organic fragment.

In [ ]:
# Identify disconnected structures (salts / multi-component entries)
data['is_salt'] = data['ROMol'].apply(is_disconnected)
print(f"Disconnected structures identified: {data['is_salt'].sum()}")

# Isolate salt rows for targeted cleaning
salts = data[data['is_salt']].copy()
PandasTools.AddMoleculeColumnToFrame(salts, 'SMILES_raw', 'ROMol')

In [ ]:
# Common counterions, solvents, and inorganic fragments to strip
FRAGMENTS_TO_REMOVE = [
    '[Na,K,Ca]',                    # alkali/alkaline earth metal ions
    '[HCl]',                        # hydrochloric acid
    'C(=CC(=O)O)C(=O)O',            # fumaric acid
    '[NH4]',                        # ammonium
    '[Cl,F,I,Br]',                  # halide ions
    'O',                            # water
    'CCO',                          # ethanol
    'CO',                           # methanol
    '[O-]S(=O)(=O)[O-]',            # sulfate
    '[N+](=O)([O-])[O-]',           # nitrate
]

In [ ]:
# Iteratively strip each listed fragment from salt structures
for fragment in FRAGMENTS_TO_REMOVE:
    salts['ROMol'] = salts['ROMol'].apply(lambda mol: remove_fragment(mol, fragment))

In [ ]:
# For any structures still disconnected after targeted stripping,
# retain only the largest fragment by atom count
salts['ROMol'] = salts['ROMol'].apply(keep_largest_fragment)

# Update SMILES_raw to reflect the cleaned salt structures
salts['SMILES_raw'] = salts['ROMol'].apply(Chem.MolToSmiles)
print(f"Salt cleaning complete. {len(salts)} entries processed.")

In [ ]:
# Merge cleaned salt entries back into the main dataframe.
# The cleaned ROMol and SMILES_raw values replace the originals at the salt indices.
data.loc[salts.index, 'SMILES_raw'] = salts['SMILES_raw']
data.loc[salts.index, 'ROMol'] = salts['ROMol']

# Regenerate the full ROMol column from the updated SMILES to ensure consistency
PandasTools.AddMoleculeColumnToFrame(data, 'SMILES_raw', 'ROMol')
print(f"Cleaned salt structures reintegrated. Dataset size: {len(data)}")

## 8. Remove Stereochemistry

In [ ]:
# Strip all stereo annotations (E/Z, R/S) to allow comparison of flat structures
data['ROMol'].apply(Chem.RemoveStereochemistry)

## 9. Neutralize Charges

In [ ]:
# Apply RDKit's Uncharger followed by atom-level neutralization
# to remove formal charges where chemically appropriate
uncharger = rdMolStandardize.Uncharger()
data['ROMol'] = data['ROMol'].apply(uncharger.uncharge)
data['ROMol'] = data['ROMol'].apply(neutralize_atoms)
print("Charge neutralization complete.")

## 10. Canonicalize Tautomers

In [ ]:
# Enumerate and select the canonical tautomer for each compound
# to ensure consistent representation of the same chemical entity
data['ROMol'] = data['ROMol'].apply(
    rdkit.Chem.MolStandardize.rdMolStandardize.CanonicalTautomer
)
print("Tautomer canonicalization complete.")

## 11. Generate Cleaned SMILES

In [ ]:
# Convert cleaned ROMol objects back to SMILES strings
data['SMILES'] = data['ROMol'].apply(Chem.MolToSmiles)
print(f"SMILES generated for {data['SMILES'].notna().sum()} rows.")

## 12. Intermediate Cleanup

In [ ]:
# Drop intermediate columns no longer needed before PubChem standardization
data.drop(
    columns=['SMILES_raw', 'ROMol', 'contains_carbon', 'is_salt'],
    inplace=True
)
print(f"Dataframe shape after cleanup: {data.shape}")
data.head()

## 13. PubChem Structure Standardization

Submit SMILES to the PubChem standardization service in batches. This applies
PubChem's own standardization rules (bond order perception, aromaticity, etc.)
to produce structures consistent with the PubChem reference database.

In [ ]:
standardized_smiles = standardize_in_batches(
    data['SMILES'].tolist(),
    initial_batch_size=1000
)
print(f"Standardization complete. {len(standardized_smiles)} SMILES returned.")

In [ ]:
# Display number of failed standardizations
fails = 0
for x in standardized_smiles:
    if x == None:
        fails += 1
print(f"Number of failed standardizations: {fails}")

In [ ]:
# Add PubChem-standardized SMILES as a new column
data['SMILES_standardized'] = standardized_smiles

In [ ]:
# Fix failed standardizations
max_iterations = 11
failed_SMILES = []
iteration = 1

while fails > 0 and iteration < max_iterations:
    print(f"Starting iteration: {iteration} ----------------------------------------------------")
    failed = data[data["SMILES_standardized"].isna()].reset_index()
    failed_SMILES = failed['SMILES']
    standardized_SMILES = standardize_in_batches(failed_SMILES, initial_batch_size=1000)
    fails = 0
    for x in standardized_SMILES:
        if x == None:
            fails += 1
    print(f"Number of failed standardizations: {fails}")
    data.loc[data['SMILES_standardized'].isna(), 'SMILES_standardized'] = standardized_SMILES 
    iteration += 1

## 14. Final Cleanup and Export

In [ ]:
# Replace pre-standardization SMILES with the PubChem-standardized version
data.drop(columns=['SMILES'], inplace=True)
data.rename(columns={'SMILES_standardized': 'SMILES'}, inplace=True)
print(f"Final dataset: {len(data)} rows.")
data.head()

In [ ]:
# Export the cleaned dataset
output_path = "your_data_with_cleaned_and_standardized_smiles.csv"
data.to_csv(output_path, index=False)
print(f"Cleaned data exported to '{output_path}'.")